# Vision QA — baseline de clasificare

**Scop.** O schiță executabilă pentru clasificatorul PASS/REJECT, ca structura să fie gata
înainte de prima antrenare serioasă.

**Intrare.** `ml/artifacts/qa/qa-frames.json`, produs de `qa-01-dataset.ipynb`.
**Ieșire.** `ml/artifacts/qa/vision-baseline.pt`.

**Atenție.** Antrenarea propriu-zisă se mută în `ml/training/vision_qa/train.py` înainte de o
rulare care contează. Notebook-ul rămâne pentru explorare și rapoarte.


## 1. Mediu și dependențe

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "ml" / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "ml" / "src").is_dir():
    raise RuntimeError("rulează notebook-ul din repository")
sys.path.insert(0, str(ROOT / "ml" / "src"))
print("repository:", ROOT)

import importlib.util

missing = [name for name in ("torch", "torchvision", "PIL") if importlib.util.find_spec(name) is None]
print("pachete lipsă:", missing)
if missing:
    raise SystemExit('rulează: pip install -e "ml[vision]"')

import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18


## 2. Încarcă manifestul

In [ ]:
ARTIFACTS = ROOT / "ml" / "artifacts" / "qa"
MANIFEST_PATH = ARTIFACTS / "qa-frames.json"
if not MANIFEST_PATH.is_file():
    raise SystemExit("rulează mai întâi qa-01-dataset.ipynb")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
rows = manifest["frames"]

train_cycles = {tuple(pair) for pair in manifest["train_cycles"]}
val_cycles = {tuple(pair) for pair in manifest["val_cycles"]}
train_rows = [row for row in rows if (row["session"], row["cycle_id"]) in train_cycles]
val_rows = [row for row in rows if (row["session"], row["cycle_id"]) in val_cycles]
print("train", len(train_rows), "val", len(val_rows))


## 3. Dataset

Eticheta este `REJECT` = 1, orice alt verdict = 0. Calea imaginii vine din manifest, deci
datasetul nu depinde de structura de directoare.


In [ ]:
class FrameDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = Image.open(row["absolute_path"]).convert("RGB")
        label = 1 if row["verdict"] == "REJECT" else 0
        return self.transform(image), torch.tensor(label, dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(FrameDataset(train_rows, train_transform), batch_size=8, shuffle=True)
val_loader = DataLoader(FrameDataset(val_rows, val_transform), batch_size=8, shuffle=False)


## 4. Model și buclă de antrenare

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += float(loss) * len(labels)
        correct += int((logits.argmax(dim=1) == labels).sum())
        total += len(labels)
    return total_loss / max(total, 1), correct / max(total, 1)


for epoch in range(5):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader, False)
    print(f"epoca {epoch + 1}: train {train_loss:.4f}/{train_acc:.2f}, val {val_loss:.4f}/{val_acc:.2f}")


## 5. Evaluare

Cu 35 de cadre `REJECT` într-o singură sesiune, cifrele sunt orientative. Raportează matricea de
confuzie și recall-ul pe `REJECT`; acuratețea singură ascunde clasa minoritară.


In [ ]:
truth = []
predicted = []
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(device))
        predicted.extend(logits.argmax(dim=1).cpu().tolist())
        truth.extend(labels.tolist())

print("pereche (adevăr, predicție):", list(zip(truth, predicted)))
print("REJECT în validare:", sum(truth), "REJECT prezis corect:", sum(1 for t, p in zip(truth, predicted) if t == 1 and p == 1))


## 6. Salvează checkpoint-ul

In [ ]:
torch.save(
    {"model": model.state_dict(), "classes": ["PASS", "REJECT"], "epochs": 5},
    ARTIFACTS / "vision-baseline.pt",
)
print("scris", ARTIFACTS / "vision-baseline.pt")


## 7. Pasul următor

1. Colectează defecte în mai multe familii și sesiuni; un singur `demo` nu susține un clasificator.
2. Split pe sesiune, apoi pe ciclu; verifică recall-ul pe `REJECT` la fiecare prag.
3. Mută antrenarea în `ml/training/vision_qa/train.py`, cu transformările versionate în cod.
4. Scrie cardul în `ml/models/vision_qa.md` înainte de orice promovare.
